In [12]:
print("Real fraud count:", y_true.sum())
print("Predicted fraud count:", y_pred.sum())


Real fraud count: 1336
Predicted fraud count: 2484


In [11]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred))


              precision    recall  f1-score   support

           0       0.88      0.76      0.82      8664
           1       0.17      0.32      0.23      1336

    accuracy                           0.70     10000
   macro avg       0.53      0.54      0.52     10000
weighted avg       0.79      0.70      0.74     10000



In [10]:
y_proba = pipeline.predict_proba(X_new)[:, 1]
y_pred = (y_proba >= threshold).astype(int)


In [8]:
X_new = df_test_new.drop("is_fraud", axis=1)
y_true = df_test_new["is_fraud"]


In [7]:
df_test_new["transaction_time"] = pd.to_datetime(df_test_new["transaction_time"])

df_test_new["hour"] = df_test_new["transaction_time"].dt.hour
df_test_new["day_of_week"] = df_test_new["transaction_time"].dt.dayofweek
df_test_new["is_weekend"] = df_test_new["day_of_week"].isin([5,6]).astype(int)

df_test_new["log_amount"] = np.log1p(df_test_new["amount"])
df_test_new["country_mismatch"] = (
    df_test_new["country"] != df_test_new["merchant_country"]
).astype(int)

df_test_new["high_amount"] = (df_test_new["amount"] > 300).astype(int)
df_test_new["is_night"] = df_test_new["hour"].between(0,5).astype(int)
df_test_new["high_velocity"] = (df_test_new["transaction_count_24h"] > 3).astype(int)

risky_categories = ["electronics", "gaming"]
df_test_new["risky_merchant"] = (
    df_test_new["merchant_category"].isin(risky_categories)
).astype(int)

df_test_new["amount_vs_avg"] = (
    df_test_new["amount"] / (df_test_new["avg_amount_24h"] + 1)
)

df_test_new["risk_combo"] = (
    df_test_new["is_international"] +
    df_test_new["is_night"] +
    df_test_new["high_velocity"]
)


In [6]:
df_test_new = pd.read_csv("new_fraud_test_data.csv")


In [5]:
import joblib
import pandas as pd
import numpy as np

pipeline = joblib.load("../Model/xgb_fraud_pipeline.pkl")
threshold = joblib.load("../Model/fraud_threshold.pkl")


In [4]:
df_new["is_fraud"].value_counts(normalize=True)


is_fraud
0    0.8664
1    0.1336
Name: proportion, dtype: float64

In [3]:
df_new.to_csv("new_fraud_test_data.csv", index=False)


In [2]:
import pandas as pd
import numpy as np

np.random.seed(999)

N = 10_000

countries = ["US", "IN", "UK", "DE", "FR", "SG"]
merchant_categories = ["electronics", "fashion", "grocery", "travel", "gaming"]
device_types = ["mobile", "web", "pos"]
channels = ["online", "card_present", "in_app"]
payment_methods = ["credit_card", "debit_card", "upi", "wallet"]
currencies = ["USD", "INR", "EUR"]

df_new = pd.DataFrame({
    "transaction_id": np.arange(100000, 100000 + N),
    "customer_id": np.random.randint(2000, 9000, N),
    "amount": np.round(np.random.exponential(scale=150, size=N), 2),
    "currency": np.random.choice(currencies, N),
    "transaction_time": pd.to_datetime("2024-04-01") +
        pd.to_timedelta(np.random.randint(0, 60*60*24*30, N), unit="s"),
    "merchant_id": np.random.randint(300, 1000, N),
    "merchant_category": np.random.choice(merchant_categories, N),
    "merchant_country": np.random.choice(countries, N),
    "country": np.random.choice(countries, N),
    "device_type": np.random.choice(device_types, N),
    "channel": np.random.choice(channels, N),
    "payment_method": np.random.choice(payment_methods, N),
    "transaction_count_24h": np.random.poisson(lam=2.5, size=N),
    "avg_amount_24h": np.round(np.random.exponential(scale=120, size=N), 2),
})

# International flag
df_new["is_international"] = (
    df_new["country"] != df_new["merchant_country"]
).astype(int)

# ---------- FRAUD LOGIC (unseen but realistic) ----------
fraud_score = (
    (df_new["amount"] > 350).astype(int) +
    df_new["is_international"] +
    (df_new["transaction_count_24h"] > 4).astype(int) +
    (df_new["device_type"] == "mobile").astype(int) +
    (df_new["channel"] == "online").astype(int) +
    df_new["merchant_category"].isin(["electronics", "gaming"]).astype(int)
)

fraud_probability = np.clip(fraud_score / 7, 0, 1)

df_new["is_fraud"] = (
    np.random.rand(N) < fraud_probability * 0.45
).astype(int)

print("New fraud rate:", df_new["is_fraud"].mean())
df_new.head()


New fraud rate: 0.1336


,transaction_id,customer_id,amount,currency,transaction_time,merchant_id,merchant_category,merchant_country,country,device_type,channel,payment_method,transaction_count_24h,avg_amount_24h,is_international,is_fraud
0,100000,7568,5.18,INR,2024-04-22 01:47:11,987,electronics,US,UK,web,in_app,wallet,2,25.43,1,0
1,100001,6444,117.19,USD,2024-04-19 18:24:11,722,electronics,DE,UK,web,in_app,wallet,1,126.11,1,0
2,100002,6965,15.48,INR,2024-04-10 17:31:11,491,grocery,IN,US,web,card_present,wallet,1,14.08,1,0
3,100003,2481,295.44,USD,2024-04-21 13:29:41,918,gaming,US,FR,mobile,online,debit_card,4,204.44,1,0
4,100004,7832,318.82,EUR,2024-04-08 09:21:58,411,fashion,IN,UK,pos,card_present,credit_card,0,77.95,1,0


In [1]:
import joblib
import pandas as pd

pipeline = joblib.load("../Model/xgb_fraud_pipeline.pkl")
threshold = joblib.load("../Model/fraud_threshold.pkl")
